In [65]:
import sys 
sys.path.append("..")
import numpy as np
import pandas as pd
from src.winners_curse import split_halves

df = pd.read_csv("../data/clean/upworthy-archive-exploratory-packages-03.12.2020 (1).csv")
clean = df[df["problem"] == 0].copy()


## Fake tests with known true rates
Simulated copies of the real tests, with a known true click rate per package, used to check the split method 

In [67]:
rng = np.random.default_rng(2) #fixed seed, same splits each run
R = 100 # number of random split, its a placeholder to check it is stable
tid = clean["clickability_test_id"].to_numpy() # test id

rows = []
for r in range(R):
    c_sel, n_sel, c_val, n_val = split_halves(clean["clicks"], clean["impressions"], rng)
    d = pd.DataFrame({"test": tid, "sel": c_sel / n_sel, "val": c_val / n_val})
    d = d.sort_values(["test", "sel"], ascending=[True, False], kind = "stable")
    d["rank"] = d.groupby("test").cumcount()
    best = d[d["rank"] == 0].set_index("test")
    runner = d[d["rank"] == 1].set_index("test")
    # lead o f best over runner up in each half, for the SAME two packages
    rows.append(pd.DataFrame({
        "lead_sel": best["sel"] - runner["sel"],
        "lead_val": best["val"] - runner["val"],
        "rep": r,}))
leads = pd.concat(rows)
curse = leads["lead_sel"] - leads["lead_val"]

print(f"mean lead, selection half:  {leads['lead_sel'].mean() * 100:.3f} pp")
print(f"mean lead, validation half: {leads['lead_val'].mean() * 100:.3f} pp")
print(f"average winner's curse:     {curse.mean() * 100:.3f} pp")
    




mean lead, selection half:  0.319 pp
mean lead, validation half: 0.160 pp
average winner's curse:     0.159 pp


In [69]:

# one split, one test
c_sel, n_sel, c_val, n_val = split_halves(clean["clicks"], clean["impressions"], np.random.default_rng(0))
print(clean[["clicks", "impressions"]].head(4).assign(c_sel=c_sel[:4], n_sel=n_sel[:4], c_val=c_val[:4], n_val=n_val[:4]))


   clicks  impressions  c_sel  n_sel  c_val  n_val
0     150         3052     71   1526     79   1526
1     122         3033     65   1516     57   1517
2     110         3092     63   1546     47   1546
3      90         3526     41   1763     49   1763


Result of real exploratory data, 100 random splits:
 - Sel half: best pkg leads runner by 0.319 pp on avg
 - Val half, same two pkgs: lead drops to 0.160 pp
 - Winner's Curse: 0.159 pp, which is about half of Sel_half lea. Which means roughly half o f the winners measured lead did not show up on fresh data

To keep in mind:
 - measured in half sized sample = noisier than if it was all of them. which can overstate the winner picked
 - we need to check on other seeds beside 100

In [71]:
rng = np.random.default_rng(6) #fixed seed, same splits each run
R = 200 # number of random split, its a placeholder to check it is stable
tid = clean["clickability_test_id"].to_numpy() # test id

rows = []
for r in range(R):
    c_sel, n_sel, c_val, n_val = split_halves(clean["clicks"], clean["impressions"], rng)
    d = pd.DataFrame({"test": tid, "sel": c_sel / n_sel, "val": c_val / n_val})
    d = d.sort_values(["test", "sel"], ascending=[True, False], kind = "stable")
    d["rank"] = d.groupby("test").cumcount()
    best = d[d["rank"] == 0].set_index("test")
    runner = d[d["rank"] == 1].set_index("test")
    # lead o f best over runner up in each half, for the SAME two packages
    rows.append(pd.DataFrame({
        "lead_sel": best["sel"] - runner["sel"],
        "lead_val": best["val"] - runner["val"],
        "rep": r,}))
leads = pd.concat(rows)
curse = leads["lead_sel"] - leads["lead_val"]

print(f"mean lead, selection half:  {leads['lead_sel'].mean() * 100:.3f} pp")
print(f"mean lead, validation half: {leads['lead_val'].mean() * 100:.3f} pp")
print(f"average winner's curse:     {curse.mean() * 100:.3f} pp")

mean lead, selection half:  0.319 pp
mean lead, validation half: 0.160 pp
average winner's curse:     0.158 pp


In [73]:
# So, how much would the avg. curse (which is now 0.158 pp) would change if we had a different set of tests?
# lets see
boot_rng = np.random.default_rng(7)
B = 2000 # number of redraws
per_test = leads.groupby(level=0)[["lead_sel", "lead_val"]].mean()
curse = (per_test["lead_sel"] - per_test["lead_val"]).to_numpy()
print("tests:", len(curse))

boot = np.array([
    boot_rng.choice(curse, size=len(curse), replace = True).mean() # we redraw one curse number per test, 3,823 in total. and get pretend curse's mean 
    for _ in range(B)
])
low, high = np.percentile(boot, [2.5, 97.5]) * 100 # middle 95% - cut off low and high 2.5% amd convert to %
print(f"mean curse: {curse.mean() * 100:.3f} pp")
print(f"95% range:  [{low:.3f}, {high:.3f}] pp")

tests: 3823
mean curse: 0.158 pp
95% range:  [0.152, 0.164] pp
